# Caracterización de las siete series con catch22

Este cuaderno cierra el Laboratorio 2 con el ejercicio de catch22. Extrae las 22 características canónicas de las siete series mensuales construidas en el Laboratorio 1, arma la matriz serie por característica y la analiza con PCA, clustering, mapa de calor, matriz de correlaciones y mapa de distancias entre series.

A diferencia de los cuadernos 09 y 10, que modelaron solo total y vía aérea sobre el conjunto de entrenamiento, aquí entran las siete series completas, de enero de 2009 a junio de 2026 (210 meses): el ejercicio es descriptivo y no de pronóstico, así que no hay partición que respetar. Produce `resultados/catch22_*.csv` y las figuras `catch22_*.png`.

## 1. La idea detrás de catch22

Comparar series por su dinámica obliga a elegir indicadores. En el cuaderno 07 los elegimos a mano: fuerza estacional, fuerza de tendencia, pendiente prepandemia, coeficiente de variación e impacto de la pandemia. Son cinco decisiones defendibles, pero arbitrarias, y nada garantiza que sean las que mejor separan estas siete series. La biblioteca `hctsa` lleva la idea al extremo opuesto y calcula miles de operaciones sobre una misma serie: exhaustivo, caro y muy redundante, porque cientos de esas operaciones miden casi lo mismo.

catch22 es el punto medio. Lubba et al. (2019) partieron de una versión filtrada de `hctsa` con 4,791 características y las evaluaron sobre 93 conjuntos de clasificación de series de tiempo, más de 147,000 series en total. Descartaron las que no superan al azar, agruparon las restantes por la similitud de su desempeño entre conjuntos —dos características que aciertan y fallan en los mismos problemas son redundantes— y conservaron un representante por grupo. De 4,791 quedaron 22. La reducción cuesta en promedio 7 % de exactitud de clasificación y devuelve un factor cercano a 1000 en tiempo de cómputo, con escalamiento casi lineal en la longitud de la serie.

Las 22 características cubren ocho familias: forma de la distribución de valores, ubicación de los eventos extremos, autocorrelación lineal, autocorrelación no lineal, contenido espectral y periodicidad, diferencias sucesivas y error de pronósticos locales, dinámica simbólica y rachas, y escalamiento de fluctuaciones. El nombre de cada una codifica la operación y sus parámetros: `CO_f1ecac` es el primer cruce de la autocorrelación por 1/e y `SB_BinaryStats_mean_longstretch1` es la racha más larga por encima de la media. La celda siguiente imprime el catálogo completo.

Un detalle que condiciona todo el ejercicio: las características se calculan sobre la serie estandarizada, así que describen forma y dinámica, no nivel ni escala. Por eso la variante `catch24` reincorpora la media y la desviación como dos características extra. Aquí se usan las 22 canónicas, que es lo que pide el enunciado.

La importancia práctica es que catch22 convierte una serie de longitud arbitraria en un vector de longitud fija e interpretable. Eso habilita el resto del ejercicio, PCA, clustering y distancias entre series, con herramienta multivariada ordinaria, y pone en el mismo plano a la serie total, con una media de 248,990 viajeros mensuales, y a vía marítima, con 5,851: la invariancia de escala evita que la magnitud domine la comparación, que es justo el problema que tuvo el comparativo del Laboratorio 1, donde cada indicador hubo que normalizarlo a mano. Y a diferencia de un vector aprendido por una red, cada coordenada tiene nombre y significado, de modo que las diferencias entre series se pueden explicar y no solo medir.

Queda una limitación declarada desde ahora: catch22 se seleccionó para clasificar series de benchmark y varias de sus características necesitan series largas. Las nuestras tienen 210 observaciones mensuales, así que las dos de escalamiento de fluctuaciones, que ajustan pendientes sobre varias escalas temporales, y las que dependen de la matriz de transición son las más expuestas a resultar inestables o constantes. El inciso 2 lo verifica antes de usarlas.

> Lubba, C. H., Sethi, S. S., Knaute, P., Schultz, S. R., Fulcher, B. D. y Jones, N. S. (2019). catch22: CAnonical Time-series CHaracteristics. *Data Mining and Knowledge Discovery*, 33(6), 1821-1852. arXiv:1901.10200.

La implementación usada es `pycatch22`, el binding oficial de la versión en C de los autores, agregado a `requirements-lab2.txt`.

In [1]:
from importlib.metadata import version
from pathlib import Path
import sys

RAIZ = Path.cwd()
if not (RAIZ / "src").exists():
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

from src.catch22 import catalogo

print(f"pycatch22 {version('pycatch22')}")

tabla = catalogo()
print(f"{len(tabla)} características en {tabla['familia'].nunique()} familias")
for familia, grupo in tabla.groupby("familia", sort=False):
    print(f"\n{familia}")
    for _, fila in grupo.iterrows():
        print(f"  {fila['caracteristica']:<44s}{fila['descripcion']}")

pycatch22 0.4.5
22 características en 8 familias

Distribución de valores
  DN_HistogramMode_5                          Moda de la distribución de valores, histograma de 5 bins
  DN_HistogramMode_10                         Moda de la distribución de valores, histograma de 10 bins

Autocorrelación lineal
  CO_f1ecac                                   Primer cruce de la ACF por 1/e
  CO_FirstMin_ac                              Retardo del primer mínimo de la ACF

Autocorrelación no lineal
  CO_HistogramAMI_even_2_5                    Información mutua con retardo 2, histograma de 5 bins
  CO_trev_1_num                               Asimetría temporal de las diferencias sucesivas (trev)
  CO_Embed2_Dist_tau_d_expfit_meandiff        Ajuste exponencial a las distancias en el espacio embebido 2-D
  IN_AutoMutualInfoStats_40_gaussian_fmmi     Primer mínimo de la información mutua, estimador gaussiano

Diferencias y pronóstico local
  MD_hrv_classic_pnn40                        Proporción de di

## 2. Extracción de las 22 características

Las características se calculan sobre la serie completa, los 210 meses de enero de 2009 a junio de 2026, y no sobre el conjunto de entrenamiento. Los cuadernos 09 y 10 respetaron la partición porque pronosticaban; aquí el objetivo es describir y comparar la dinámica de las siete series, no predecir, así que apartar 63 meses solo desperdiciaría información. Las series por país de residencia y vía marítima tienen meses en cero, y por eso quedaron fuera del modelado LSTM, que dependía de `log1p`; catch22 no necesita esa transformación, así que las siete entran completas.

`extraer_serie` llama a `pycatch22.catch22_all` y devuelve las 22 características indexadas por nombre, después de comprobar que la biblioteca las entregó en el orden del catálogo. La comprobación no es adorno: el catálogo del inciso 1 asigna familia y descripción por nombre, de modo que un reordenamiento en una versión distinta de `pycatch22` dejaría la matriz mal etiquetada sin que nada fallara.

La celda siguiente verifica además tres cosas antes de que el resto del cuaderno dependa de ellas.

- **Invariancia de escala.** `catch22_all` estandariza la serie internamente, así que `2 · serie + 1000` devuelve exactamente los mismos 22 valores. Eso es lo que hace comparable a la serie total, con media de 248,990 viajeros mensuales, con vía marítima, con 5,851, y lo que implica que ni el nivel ni la dispersión entran en la matriz: lo que queda es forma y dinámica.
- **Ausencia de valores faltantes.** Ninguna de las 154 celdas resulta NaN, ni en las series con meses en cero.
- **Variabilidad entre series.** Ninguna de las 22 características toma el mismo valor en las siete series. Esto matiza la limitación anunciada en el inciso 1: incluso las dos de escalamiento de fluctuaciones, que son las que más longitud exigen, discriminan entre series. Las 22 se conservan, y el inciso 4 podrá estandarizarlas por columna sin divisiones entre cero.

In [2]:
import numpy as np
import pandas as pd

from src.catch22 import extraer_serie
from src.utils import SERIES, cargar_serie

completas = {clave: cargar_serie(clave, "completa") for clave in SERIES}
extraidas = pd.DataFrame(
    {clave: extraer_serie(serie) for clave, serie in completas.items()}
).T

assert extraidas.shape == (len(SERIES), len(tabla))
assert np.allclose(extraidas.loc["total"], extraer_serie(2 * completas["total"] + 1000))

meses = sorted({len(serie) for serie in completas.values()})
constantes = list(extraidas.columns[extraidas.std(ddof=0) == 0])
print(f"series: {extraidas.shape[0]}, características: {extraidas.shape[1]}")
print(f"meses por serie: {meses}")
print(f"NaN en la extracción: {int(extraidas.isna().sum().sum())}")
print(f"características constantes entre series: {constantes or 'ninguna'}")
print("invariancia de escala: 2 · serie + 1000 devuelve los mismos 22 valores")

catalogo_indexado = tabla.set_index("caracteristica")
ejemplo = pd.DataFrame(
    {
        "valor": extraidas.loc["total"].round(4),
        "familia": catalogo_indexado["familia"],
        "descripcion": catalogo_indexado["descripcion"],
    }
)
print(f"\nvector de la serie total\n{ejemplo.to_string()}")

series: 7, características: 22
meses por serie: [210]
NaN en la extracción: 0
características constantes entre series: ninguna
invariancia de escala: 2 · serie + 1000 devuelve los mismos 22 valores

vector de la serie total
                                               valor                         familia                                                          descripcion
DN_HistogramMode_5                            0.1885         Distribución de valores             Moda de la distribución de valores, histograma de 5 bins
DN_HistogramMode_10                          -0.5804         Distribución de valores            Moda de la distribución de valores, histograma de 10 bins
CO_f1ecac                                     8.5789          Autocorrelación lineal                                       Primer cruce de la ACF por 1/e
CO_FirstMin_ac                               10.0000          Autocorrelación lineal                                  Retardo del primer mínimo de la ACF
CO_His

## 3. Matriz serie por característica

La matriz lleva las siete series en las filas y las 22 características en las columnas, que es la forma que pide el enunciado y también la que espera `scikit-learn`: una observación por fila. `matriz_caracteristicas` devuelve solo el bloque numérico, indexado por la clave de la serie. La etiqueta y la categoría se agregan al escribir `resultados/catch22_caracteristicas.csv`, con el mismo formato de `comparativo_series.csv` del Laboratorio 1, y las categorías se reutilizan de `src/comparativo.py` en lugar de redefinirlas, para que el inciso 10, que pregunta si las series de una misma categoría se agrupan, use exactamente la clasificación del comparativo anterior.

La matriz se imprime transpuesta, con las características en las filas, porque 22 columnas no caben legibles a lo ancho; es el mismo recurso que usa el cuaderno 07 para los perfiles estacionales.

Así impresa se ve el problema que resuelve el inciso 4: las columnas viven en escalas incomparables. `SB_BinaryStats_mean_longstretch1` va de 9 a 50 meses y `PD_PeriodicityWang_th0_01` de 2 a 11, mientras `SB_TransitionMatrix_3ac_sumdiagcov` se mueve entre 0.006 y 0.111. Una distancia euclidiana sobre la matriz cruda quedaría decidida por dos o tres columnas y las diecinueve restantes no aportarían nada.

In [3]:
from src.catch22 import matriz_caracteristicas
from src.comparativo import CATEGORIAS
from src.utils import RUTA_RESULTADOS

matriz = matriz_caracteristicas(completas)

assert list(matriz.index) == list(SERIES)
assert list(matriz.columns) == list(tabla["caracteristica"])

exportable = matriz.reset_index()
exportable.insert(1, "etiqueta", [SERIES[clave] for clave in matriz.index])
exportable.insert(2, "categoria", [CATEGORIAS[clave] for clave in matriz.index])
exportable.to_csv(RUTA_RESULTADOS / "catch22_caracteristicas.csv", index=False)

print(f"matriz {matriz.shape[0]} x {matriz.shape[1]} en resultados/catch22_caracteristicas.csv")
print(matriz.T.round(3).to_string())

rangos = (matriz.max() - matriz.min()).sort_values(ascending=False)
print("\nrecorrido de cada característica entre las siete series")
print(rangos.round(3).to_string())

matriz 7 x 22 en resultados/catch22_caracteristicas.csv
clave                                         total  via_aerea  via_terrestre  via_maritima  pais_el_salvador  pais_estados_unidos  pais_honduras
DN_HistogramMode_5                            0.189     -0.408         -0.547        -0.432            -0.367               -0.629         -0.222
DN_HistogramMode_10                          -0.580     -0.123         -0.782        -0.652            -0.590               -0.373         -0.432
CO_f1ecac                                     8.579      6.437          8.997         2.771            13.874                9.224         18.925
CO_FirstMin_ac                               10.000     10.000          3.000         6.000             3.000                2.000          8.000
CO_HistogramAMI_even_2_5                      0.454      0.207          0.537         0.123             0.476                0.229          0.477
CO_trev_1_num                                -0.127     -0.175      